# Teste de hiperparâmetros do FAGTB

Este notebook testa um produto cartesiano de hiperparâmetros do FAGTB usando o mesmo wrapper `modelo_fagtb` do fluxo principal.

A ideia é usar uma amostra menor do dataset, por exemplo 5.000 linhas do HMDA, para encontrar uma região razoável de hiperparâmetros antes de rodar o fluxo principal completo.


In [1]:
# ============================================================
# CONFIGURAÇÕES GERAIS
# ============================================================

from pathlib import Path

# Raiz do projeto
PROJECT_ROOT = Path.cwd()

# Dataset que será usado para escolher hiperparâmetros
CAMINHO_DATASET = Path("datasets/tratado/hmda.csv")

# Nome do dataset dentro do configs_datasets.json
# Deve bater com uma chave do JSON, por exemplo: "hmda" ou "compas"
NOME_DATASET = "hmda"

# Arquivo de configuração dos datasets
CAMINHO_CONFIGS = Path("configs_datasets.json")

# Pasta de saída dos resultados
PASTA_SAIDA = Path("resultados/testes_hiperparametros_fagtb")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Amostragem do dataset
# ------------------------------------------------------------

# Opção 1: usar número fixo de linhas.
# Para HMDA, recomendo começar com 5000.
N_LINHAS_AMOSTRA = 5000

# Opção 2: usar uma fração do dataset.
# Só será usada se N_LINHAS_AMOSTRA = None.
FRACAO_DATASET = None

# Amostragem estratificada pelo target
AMOSTRAGEM_ESTRATIFICADA = True

# ------------------------------------------------------------
# Split interno para validação dos hiperparâmetros
# ------------------------------------------------------------

TEST_SIZE_VALIDACAO = 0.30
RANDOM_STATE = 42

# ------------------------------------------------------------
# Uso do atributo sensível como feature
# ------------------------------------------------------------

# True  -> cenário aware: mantém o atributo sensível no X
# False -> cenário unaware: remove o atributo sensível do X antes do treino
USAR_ATRIBUTO_SENSIVEL_NO_MODELO = True

# ------------------------------------------------------------
# Controle de execução
# ------------------------------------------------------------

# Use True para testar só a primeira configuração sensível do JSON.
# Ajuda a estimar tempo antes de rodar tudo.
TESTAR_APENAS_PRIMEIRA_CONFIG_SENSIVEL = False

# Quantos primeiros testes usar para estimar tempo total durante a execução.
N_TESTES_PARA_ESTIMAR_TEMPO = 3

# Salvar os modelos de cada combinação geralmente pesa muito.
# Recomendo False para busca de hiperparâmetros.
SALVAR_MODELOS = False

# ------------------------------------------------------------
# Critério de escolha
# ------------------------------------------------------------

# O score abaixo tenta escolher modelos que aumentem balanced accuracy,
# sem ignorar completamente justiça. Ajuste a penalidade se quiser.
PENALIDADE_EQUALIZED_ODDS = 0.05
PENALIDADE_EQUAL_OPPORTUNITY = 0.03

# ------------------------------------------------------------
# Grade de hiperparâmetros
# ------------------------------------------------------------

GRADE_HIPERPARAMETROS = {
    "lambda_fagtb": [ 0.0001, 0.0005, 0.001, 0.005, 0.01],
    "n_estimators": [100, 200],
    "learning_rate": [0.03, 0.05, 0.10],
    "max_depth": [3, 5],
    "max_features": [None],
    "min_samples_split": [2],
    "min_impurity": [0],
    "regression": [False],
}


In [2]:
# ============================================================
# IMPORTS
# ============================================================

import sys
import gc
import json
import time
import itertools
import importlib
import numpy as np
import pandas as pd
import joblib

from tqdm import tqdm
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

sys.path.append(str(PROJECT_ROOT))

import func_aux
importlib.reload(func_aux)
from func_aux import *

print("Imports concluídos.")


Arquivo exportado: func_aux.py
Arquivo exportado: func_aux.py
Imports concluídos.


In [3]:
# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def formatar_numero(n):
    return f"{int(n):,}".replace(",", ".")


def preparar_para_fagtb(X, y, sensitive):
    """
    Converte dados para arrays compactos.
    Isso reduz bastante problemas de memória no HMDA.
    """
    if hasattr(X, "to_numpy"):
        X_np = X.to_numpy(dtype=np.float32, copy=True)
    else:
        X_np = np.asarray(X, dtype=np.float32)

    y_np = np.asarray(y, dtype=np.int32).ravel()
    sensitive_np = np.asarray(sensitive, dtype=np.int32).ravel()

    return X_np, y_np, sensitive_np


def taxa_positiva(y_pred, sensitive, grupo):
    mask = sensitive == grupo
    if mask.sum() == 0:
        return np.nan
    return np.mean(y_pred[mask] == 1)


def tpr_por_grupo(y_true, y_pred, sensitive, grupo):
    mask = (sensitive == grupo) & (y_true == 1)
    if mask.sum() == 0:
        return np.nan
    return np.mean(y_pred[mask] == 1)


def fpr_por_grupo(y_true, y_pred, sensitive, grupo):
    mask = (sensitive == grupo) & (y_true == 0)
    if mask.sum() == 0:
        return np.nan
    return np.mean(y_pred[mask] == 1)


def avaliar_modelo_fairness(
    y_true,
    y_pred,
    sensitive,
    grupo_privilegiado,
    grupo_desprivilegiado
):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_pred = np.asarray(y_pred).astype(int).ravel()
    sensitive = np.asarray(sensitive).astype(int).ravel()

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    tpr_priv = tpr_por_grupo(y_true, y_pred, sensitive, grupo_privilegiado)
    tpr_despriv = tpr_por_grupo(y_true, y_pred, sensitive, grupo_desprivilegiado)
    fpr_priv = fpr_por_grupo(y_true, y_pred, sensitive, grupo_privilegiado)
    fpr_despriv = fpr_por_grupo(y_true, y_pred, sensitive, grupo_desprivilegiado)

    taxa_pos_priv = taxa_positiva(y_pred, sensitive, grupo_privilegiado)
    taxa_pos_despriv = taxa_positiva(y_pred, sensitive, grupo_desprivilegiado)

    equal_opportunity = abs(tpr_despriv - tpr_priv)
    equalized_odds = max(
        abs(tpr_despriv - tpr_priv),
        abs(fpr_despriv - fpr_priv)
    )
    demographic_parity = abs(taxa_pos_despriv - taxa_pos_priv)

    if pd.isna(taxa_pos_priv) or pd.isna(taxa_pos_despriv) or taxa_pos_priv == 0:
        p_rule = np.nan
    else:
        p_rule = 100 * min(
            taxa_pos_despriv / taxa_pos_priv,
            taxa_pos_priv / taxa_pos_despriv if taxa_pos_despriv != 0 else np.nan
        )

    return {
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "equal_opportunity": equal_opportunity,
        "equalized_odds": equalized_odds,
        "demographic_parity": demographic_parity,
        "p_rule": p_rule,
        "tpr_privilegiado": tpr_priv,
        "tpr_desprivilegiado": tpr_despriv,
        "fpr_privilegiado": fpr_priv,
        "fpr_desprivilegiado": fpr_despriv,
        "taxa_positiva_privilegiado": taxa_pos_priv,
        "taxa_positiva_desprivilegiado": taxa_pos_despriv,
    }


def calcular_score(linha):
    return (
        linha["balanced_accuracy"]
        - PENALIDADE_EQUALIZED_ODDS * linha["equalized_odds"]
        - PENALIDADE_EQUAL_OPPORTUNITY * linha["equal_opportunity"]
    )


def resumo_melhor_valor_por_hiperparametro(df_resultados, colunas_parametros):
    """
    Para cada hiperparâmetro, calcula o melhor valor médio considerando o score.
    Isso não substitui a melhor combinação global, mas ajuda a interpretar tendências.
    """
    linhas = []

    for parametro in colunas_parametros:
        df_param = (
            df_resultados
            .groupby(parametro, dropna=False)
            .agg(
                score_medio=("score", "mean"),
                score_max=("score", "max"),
                balanced_accuracy_media=("balanced_accuracy", "mean"),
                balanced_accuracy_max=("balanced_accuracy", "max"),
                accuracy_media=("accuracy", "mean"),
                equalized_odds_media=("equalized_odds", "mean"),
                tempo_min_medio=("tempo_min", "mean"),
                n_testes=("score", "count")
            )
            .reset_index()
            .sort_values(
                by=["score_medio", "balanced_accuracy_media"],
                ascending=[False, False]
            )
        )

        melhor = df_param.iloc[0].to_dict()
        melhor["hiperparametro"] = parametro
        melhor["melhor_valor"] = melhor[parametro]
        linhas.append(melhor)

    return pd.DataFrame(linhas)


In [4]:
# ============================================================
# CARREGA DATASET E CONFIGURAÇÕES
# ============================================================

with open(CAMINHO_CONFIGS, "r", encoding="utf-8") as f:
    configs = json.load(f)

config_dataset = configs[NOME_DATASET]
configs_sensiveis = config_dataset["configs_sensiveis"]
target_col = config_dataset["target"]

if TESTAR_APENAS_PRIMEIRA_CONFIG_SENSIVEL:
    configs_sensiveis = configs_sensiveis[:1]

print(f"Dataset: {NOME_DATASET}")
print(f"Arquivo: {CAMINHO_DATASET}")
print(f"Target: {target_col}")
print(f"Configurações sensíveis: {[c['nome'] for c in configs_sensiveis]}")

# Carrega dataset completo
# Observação: se der problema de memória nesta leitura, gere antes um hmda_5pct.csv e use esse caminho.
df_original = pd.read_csv(CAMINHO_DATASET)

n_original = len(df_original)
print(f"\nLinhas no dataset original: {formatar_numero(n_original)}")

# ------------------------------------------------------------
# Amostragem
# ------------------------------------------------------------

if N_LINHAS_AMOSTRA is not None:
    n_amostra = min(N_LINHAS_AMOSTRA, n_original)
    train_size = n_amostra
elif FRACAO_DATASET is not None:
    n_amostra = int(n_original * FRACAO_DATASET)
    train_size = FRACAO_DATASET
else:
    n_amostra = n_original
    train_size = None

if n_amostra < n_original:
    if AMOSTRAGEM_ESTRATIFICADA:
        df_amostra, _ = train_test_split(
            df_original,
            train_size=n_amostra if N_LINHAS_AMOSTRA is not None else train_size,
            random_state=RANDOM_STATE,
            stratify=df_original[target_col]
        )
    else:
        df_amostra = df_original.sample(
            n=n_amostra,
            random_state=RANDOM_STATE
        )

    df_amostra = df_amostra.reset_index(drop=True)
else:
    df_amostra = df_original.reset_index(drop=True)

# Libera dataset completo para economizar memória
del df_original
gc.collect()

print(f"Linhas usadas para busca de hiperparâmetros: {formatar_numero(len(df_amostra))}")
print(f"Percentual usado: {100 * len(df_amostra) / n_original:.4f}%")

print("\nDistribuição do target na amostra:")
print(df_amostra[target_col].value_counts(normalize=True))


Dataset: hmda
Arquivo: datasets\tratado\hmda.csv
Target: Favorable Outcome (Credit Approved)
Configurações sensíveis: ['african_american_0', 'caucasian_1']

Linhas no dataset original: 34.717
Linhas usadas para busca de hiperparâmetros: 5.000
Percentual usado: 14.4022%

Distribuição do target na amostra:
Favorable Outcome (Credit Approved)
1    0.7812
0    0.2188
Name: proportion, dtype: float64


In [5]:
# ============================================================
# SPLIT TREINO / VALIDAÇÃO
# ============================================================

X = df_amostra.drop(columns=[target_col]).reset_index(drop=True)
y = df_amostra[target_col].astype(int).reset_index(drop=True)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=TEST_SIZE_VALIDACAO,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)

print("\nDistribuição y_train:")
print(y_train.value_counts(normalize=True))

print("\nDistribuição y_val:")
print(y_val.value_counts(normalize=True))

# Libera a amostra original após split
del df_amostra, X, y
gc.collect()


X_train: (3500, 50)
X_val: (1500, 50)
y_train: (3500,)
y_val: (1500,)

Distribuição y_train:
Favorable Outcome (Credit Approved)
1    0.781143
0    0.218857
Name: proportion, dtype: float64

Distribuição y_val:
Favorable Outcome (Credit Approved)
1    0.781333
0    0.218667
Name: proportion, dtype: float64


0

In [6]:
# ============================================================
# PRODUTO CARTESIANO DE HIPERPARÂMETROS
# ============================================================

grade = list(ParameterGrid(GRADE_HIPERPARAMETROS))

print(f"Combinações por configuração sensível: {len(grade)}")
print(f"Configurações sensíveis: {len(configs_sensiveis)}")
print(f"Total de treinos FAGTB: {len(grade) * len(configs_sensiveis)}")

pd.DataFrame(grade).head()


Combinações por configuração sensível: 60
Configurações sensíveis: 2
Total de treinos FAGTB: 120


,lambda_fagtb,learning_rate,max_depth,max_features,min_impurity,min_samples_split,n_estimators,regression
0,0.0001,0.03,3,None,0,2,100,False
1,0.0001,0.03,3,None,0,2,200,False
2,0.0001,0.03,5,None,0,2,100,False
3,0.0001,0.03,5,None,0,2,200,False
4,0.0001,0.05,3,None,0,2,100,False


In [7]:
# ============================================================
# BUSCA DE HIPERPARÂMETROS
# ============================================================

resultados = []
colunas_parametros = list(GRADE_HIPERPARAMETROS.keys())

total_treinos = len(grade) * len(configs_sensiveis)
contador_treinos = 0
tempos_primeiros_testes = []

for config in configs_sensiveis:

    nome_config = config["nome"]
    atributo_sensivel = config["atributo_sensivel"]
    grupo_privilegiado = config["grupo_privilegiado"]
    grupo_desprivilegiado = config["grupo_desprivilegiado"]

    print("\n" + "=" * 100, flush=True)
    print(f"Configuração sensível: {nome_config}", flush=True)
    print(f"Atributo sensível: {atributo_sensivel}", flush=True)
    print("=" * 100, flush=True)

    s_train = X_train[atributo_sensivel].astype(int).reset_index(drop=True)
    s_val = X_val[atributo_sensivel].astype(int).reset_index(drop=True)

    if USAR_ATRIBUTO_SENSIVEL_NO_MODELO:
        X_train_model = X_train.copy()
        X_val_model = X_val.copy()
        cenario = "aware"
    else:
        X_train_model = X_train.drop(columns=[atributo_sensivel]).copy()
        X_val_model = X_val.drop(columns=[atributo_sensivel]).copy()
        cenario = "unaware"

    X_train_np, y_train_np, s_train_np = preparar_para_fagtb(
        X_train_model,
        y_train,
        s_train
    )

    X_val_np, y_val_np, s_val_np = preparar_para_fagtb(
        X_val_model,
        y_val,
        s_val
    )

    for params in tqdm(grade, desc=f"FAGTB {nome_config}", unit="modelo"):

        contador_treinos += 1

        lambda_fagtb = params["lambda_fagtb"]

        print("\n" + "-" * 100, flush=True)
        print(
            f"Treino {contador_treinos}/{total_treinos} | "
            f"config={nome_config} | "
            f"lambda={lambda_fagtb} | "
            f"n_estimators={params['n_estimators']} | "
            f"learning_rate={params['learning_rate']} | "
            f"max_depth={params['max_depth']}",
            flush=True
        )
        print("-" * 100, flush=True)

        inicio = time.time()

        try:
            modelo = modelo_fagtb(
                X_train=X_train_np,
                y_train=y_train_np,
                sensitive_train=s_train_np,
                X_test=X_val_np,
                y_test=y_val_np,
                sensitive_test=s_val_np,
                lambda_fagtb=lambda_fagtb,
                n_estimators=params["n_estimators"],
                learning_rate=params["learning_rate"],
                max_depth=params["max_depth"],
                max_features=params["max_features"],
                min_samples_split=params["min_samples_split"],
                min_impurity=params["min_impurity"],
                regression=params["regression"]
            )

            y_pred_val = modelo.predict(X_val_np)

            metricas = avaliar_modelo_fairness(
                y_true=y_val_np,
                y_pred=y_pred_val,
                sensitive=s_val_np,
                grupo_privilegiado=grupo_privilegiado,
                grupo_desprivilegiado=grupo_desprivilegiado
            )

            status = "ok"
            erro = None

            if SALVAR_MODELOS:
                nome_modelo = (
                    f"modelo_{NOME_DATASET}_{nome_config}_"
                    f"lambda_{lambda_fagtb}_"
                    f"nest_{params['n_estimators']}_"
                    f"lr_{params['learning_rate']}_"
                    f"depth_{params['max_depth']}.joblib"
                )
                joblib.dump(modelo, PASTA_SAIDA / nome_modelo)

        except MemoryError as e:
            metricas = {}
            status = "MemoryError"
            erro = str(e)
            print("MemoryError nesta combinação. Pulando.", flush=True)

        except Exception as e:
            metricas = {}
            status = "erro"
            erro = repr(e)
            print(f"Erro nesta combinação: {erro}", flush=True)

        fim = time.time()
        tempo_seg = fim - inicio
        tempo_min = tempo_seg / 60

        linha = {
            "dataset": NOME_DATASET,
            "config_sensivel": nome_config,
            "atributo_sensivel": atributo_sensivel,
            "cenario": cenario,
            **params,
            "status": status,
            "erro": erro,
            "tempo_seg": tempo_seg,
            "tempo_min": tempo_min,
            **metricas
        }
        print(f"Tempo de execução: {tempo_min:.2f} min", flush=True)
        print(f"Resultado parcial: {linha}", flush=True)

        if status == "ok":
            linha["score"] = calcular_score(linha)
        else:
            linha["score"] = np.nan

        resultados.append(linha)

        df_resultados = pd.DataFrame(resultados)
        df_resultados.to_csv(PASTA_SAIDA / "resultados_hiperparametros_fagtb.csv", index=False)

        if status == "ok":
            print(
                f"Resultado | "
                f"acc={linha['accuracy']:.4f} | "
                f"bal_acc={linha['balanced_accuracy']:.4f} | "
                f"f1={linha['f1']:.4f} | "
                f"EOdds={linha['equalized_odds']:.4f} | "
                f"EOp={linha['equal_opportunity']:.4f} | "
                f"p-rule={linha['p_rule']:.2f} | "
                f"score={linha['score']:.4f} | "
                f"tempo={tempo_min:.2f} min",
                flush=True
            )

        if len(tempos_primeiros_testes) < N_TESTES_PARA_ESTIMAR_TEMPO and status == "ok":
            tempos_primeiros_testes.append(tempo_min)

            media_tempo = np.mean(tempos_primeiros_testes)
            estimativa_total_min = media_tempo * total_treinos
            estimativa_restante_min = media_tempo * (total_treinos - contador_treinos)

            print(
                f"Estimativa parcial | "
                f"média={media_tempo:.2f} min/modelo | "
                f"total≈{estimativa_total_min/60:.2f} h | "
                f"restante≈{estimativa_restante_min/60:.2f} h",
                flush=True
            )

        if 'modelo' in locals():
            del modelo
        gc.collect()

    del X_train_np, X_val_np, y_train_np, y_val_np, s_train_np, s_val_np
    gc.collect()

print("\nBusca concluída.")



Configuração sensível: african_american_0
Atributo sensível: Applicant_African-American


FAGTB african_american_0:   0%|          | 0/60 [00:00<?, ?modelo/s]


----------------------------------------------------------------------------------------------------
Treino 1/120 | config=african_american_0 | lambda=0.0001 | n_estimators=100 | learning_rate=0.03 | max_depth=3
----------------------------------------------------------------------------------------------------


0 945.2967211235476 1830.1451554416485 1830.239685113761 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
5 1700.8075461358974 1787.1979139244243 1787.367994679038 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
10 582.6756612492989 1747.0721399612758 1747.1304075274006 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
15 381.40400250799985 1707.7994375287203 1707.837577928971 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
20 2707.676071085536 1668.9083184157719 1669.1790860228805 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
25 1507.5044326867999 1631.81666

FAGTB african_american_0:   2%|▏         | 1/60 [00:12<12:36, 12.83s/modelo]


----------------------------------------------------------------------------------------------------
Treino 2/120 | config=african_american_0 | lambda=0.0001 | n_estimators=200 | learning_rate=0.03 | max_depth=3
----------------------------------------------------------------------------------------------------
0 945.2967211235476 1830.1451554416485 1830.239685113761 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
5 1700.8075461358974 1787.1979139244243 1787.367994679038 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
10 582.6756612492989 1747.0721399612758 1747.1304075274006 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
15 381.40400250799985 1707.7994375287203 1707.837577928971 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
20 2707.676071085536 1668.9083184157719 1669.1790860228805 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
25 1507.5044326867999 1631.8166650

...


----------------------------------------------------------------------------------------------------
Treino 119/120 | config=caucasian_1 | lambda=0.01 | n_estimators=100 | learning_rate=0.1 | max_depth=5
----------------------------------------------------------------------------------------------------
0 155.0059601682837 1810.1981427672456 1811.7482023689286 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
5 2094.078574199139 1674.9273720963126 1695.868157838304 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
10 2509.2970002395846 1559.8000935935884 1584.8930635959841 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
15 917.9300948543536 1464.2633090044408 1473.4426099529842 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
20 3654.632208084064 1385.4820598922943 1422.0283819731349 Accuracy: 0.8631  test :  0.84  Prule Train :  0.9581730072160695  Prule test :  0.9611522830504423
25 -84.596103

FAGTB caucasian_1:  98%|█████████▊| 59/60 [05:14<00:05,  5.05s/modelo]


----------------------------------------------------------------------------------------------------
Treino 120/120 | config=caucasian_1 | lambda=0.01 | n_estimators=200 | learning_rate=0.1 | max_depth=5
----------------------------------------------------------------------------------------------------
0 155.0059601682837 1810.1981427672456 1811.7482023689286 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
5 2094.078574199139 1674.9273720963126 1695.868157838304 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
10 2509.2970002395846 1559.8000935935884 1584.8930635959841 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
15 917.9300948543536 1464.2633090044408 1473.4426099529842 Accuracy: 0.7811  test :  0.7813  Prule Train :  1.0  Prule test :  1.0
20 3654.632208084064 1385.4820598922943 1422.0283819731349 Accuracy: 0.8631  test :  0.84  Prule Train :  0.9581730072160695  Prule test :  0.9611522830504423
25 -84.596103

FAGTB caucasian_1: 100%|██████████| 60/60 [05:21<00:00,  5.36s/modelo]


Busca concluída.


In [8]:
# ============================================================
# RESULTADOS: MELHOR COMBINAÇÃO GLOBAL POR CONFIGURAÇÃO SENSÍVEL
# ============================================================

df_resultados = pd.read_csv(PASTA_SAIDA / "resultados_hiperparametros_fagtb.csv")

df_ok = df_resultados[df_resultados["status"] == "ok"].copy()

if df_ok.empty:
    raise RuntimeError("Nenhum teste foi concluído com status ok.")

# Melhor combinação global por config sensível
idx_melhores = (
    df_ok
    .sort_values(
        by=["config_sensivel", "score", "balanced_accuracy", "accuracy", "equalized_odds"],
        ascending=[True, False, False, False, True]
    )
    .groupby("config_sensivel")
    .head(1)
    .index
)

df_melhores = df_ok.loc[idx_melhores].copy()

df_melhores = df_melhores.sort_values("config_sensivel")
df_melhores.to_csv(PASTA_SAIDA / "melhores_combinacoes_fagtb.csv", index=False)

colunas_exibir = [
    "dataset",
    "config_sensivel",
    "cenario",
    "lambda_fagtb",
    "n_estimators",
    "learning_rate",
    "max_depth",
    "max_features",
    "min_samples_split",
    "min_impurity",
    "accuracy",
    "balanced_accuracy",
    "f1",
    "equalized_odds",
    "equal_opportunity",
    "p_rule",
    "score",
    "tempo_min"
]

print("Melhores combinações por configuração sensível:")
df_melhores[colunas_exibir]


Melhores combinações por configuração sensível:


,dataset,config_sensivel,cenario,lambda_fagtb,n_estimators,learning_rate,max_depth,max_features,min_samples_split,min_impurity,accuracy,balanced_accuracy,f1,equalized_odds,equal_opportunity,p_rule,score,tempo_min
57,hmda,african_american_0,aware,0.01,200,0.1,3,NaN,2,0,0.844667,0.68763,0.906763,0.014118,0.014118,96.146734,0.686501,0.114839
117,hmda,caucasian_1,aware,0.01,200,0.1,3,NaN,2,0,0.846667,0.68891,0.908074,0.009821,0.005833,96.520883,0.688244,0.115140


In [9]:
# ============================================================
# RESULTADOS: MELHOR VALOR MÉDIO PARA CADA HIPERPARÂMETRO
# ============================================================

resumos_por_config = []

for nome_config, df_config in df_ok.groupby("config_sensivel"):
    df_resumo_param = resumo_melhor_valor_por_hiperparametro(
        df_config,
        colunas_parametros=colunas_parametros
    )
    df_resumo_param["config_sensivel"] = nome_config
    resumos_por_config.append(df_resumo_param)

df_melhor_por_parametro = pd.concat(resumos_por_config, ignore_index=True)
df_melhor_por_parametro.to_csv(
    PASTA_SAIDA / "melhor_valor_medio_por_hiperparametro.csv",
    index=False
)

print("Melhor valor médio por hiperparâmetro e configuração sensível:")
df_melhor_por_parametro[[
    "config_sensivel",
    "hiperparametro",
    "melhor_valor",
    "score_medio",
    "balanced_accuracy_media",
    "accuracy_media",
    "equalized_odds_media",
    "tempo_min_medio",
    "n_testes"
]]


Melhor valor médio por hiperparâmetro e configuração sensível:


,config_sensivel,hiperparametro,melhor_valor,score_medio,balanced_accuracy_media,accuracy_media,equalized_odds_media,tempo_min_medio,n_testes
0,african_american_0,lambda_fagtb,0.01,0.667955,0.670229,0.843778,0.036297,0.087015,12.0
1,african_american_0,n_estimators,200.0,0.673706,0.675807,0.842889,0.030580,0.115840,30.0
2,african_american_0,learning_rate,0.1,0.678299,0.680703,0.842933,0.036074,0.087367,20.0
3,african_american_0,max_depth,3.0,0.667021,0.669526,0.842622,0.039500,0.092838,30.0
4,african_american_0,max_features,NaN,0.667021,0.669526,0.842622,0.039500,0.090184,60.0
5,african_american_0,min_samples_split,2.0,0.667021,0.669526,0.842622,0.039500,0.090184,60.0
6,african_american_0,min_impurity,0.0,0.667021,0.669526,0.842622,0.039500,0.090184,60.0
7,african_american_0,regression,False,0.667021,0.669526,0.842622,0.039500,0.090184,60.0
8,caucasian_1,lambda_fagtb,0.01,0.670404,0.671926,0.845000,0.026741,0.087092,12.0
9,caucasian_1,n_estimators,200.0,0.672739,0.674340,0.842311,0.027942,0.115469,30.0


In [10]:
# ============================================================
# TOP 20 COMBINAÇÕES
# ============================================================

df_top20 = (
    df_ok
    .sort_values(
        by=["score", "balanced_accuracy", "accuracy", "equalized_odds"],
        ascending=[False, False, False, True]
    )
    .head(20)
)

df_top20.to_csv(PASTA_SAIDA / "top20_hiperparametros_fagtb.csv", index=False)

df_top20[colunas_exibir]


,dataset,config_sensivel,cenario,lambda_fagtb,n_estimators,learning_rate,max_depth,max_features,min_samples_split,min_impurity,accuracy,balanced_accuracy,f1,equalized_odds,equal_opportunity,p_rule,score,tempo_min
117,hmda,caucasian_1,aware,0.0100,200,0.10,3,NaN,2,0,0.846667,0.688910,0.908074,0.009821,0.005833,96.520883,0.688244,0.115140
119,hmda,caucasian_1,aware,0.0100,200,0.10,5,NaN,2,0,0.846667,0.688910,0.908074,0.009821,0.005833,96.520883,0.688244,0.114896
57,hmda,african_american_0,aware,0.0100,200,0.10,3,NaN,2,0,0.844667,0.687630,0.906763,0.014118,0.014118,96.146734,0.686501,0.114839
59,hmda,african_american_0,aware,0.0100,200,0.10,5,NaN,2,0,0.844667,0.687630,0.906763,0.014118,0.014118,96.146734,0.686501,0.114838
21,hmda,african_american_0,aware,0.0005,200,0.10,3,NaN,2,0,0.842667,0.687448,0.905373,0.036638,0.024036,94.625425,0.684895,0.114837
23,hmda,african_american_0,aware,0.0005,200,0.10,5,NaN,2,0,0.842667,0.687448,0.905373,0.036638,0.024036,94.625425,0.684895,0.115356
9,hmda,african_american_0,aware,0.0001,200,0.10,3,NaN,2,0,0.842000,0.684826,0.905086,0.030532,0.029935,94.345468,0.682401,0.118119
11,hmda,african_american_0,aware,0.0001,200,0.10,5,NaN,2,0,0.842000,0.684826,0.905086,0.030532,0.029935,94.345468,0.682401,0.116074
45,hmda,african_american_0,aware,0.0050,200,0.10,3,NaN,2,0,0.843333,0.684581,0.906038,0.034842,0.023693,94.765277,0.682128,0.115127
47,hmda,african_american_0,aware,0.0050,200,0.10,5,NaN,2,0,0.843333,0.684581,0.906038,0.034842,0.023693,94.765277,0.682128,0.114858


In [11]:
# ============================================================
# SUGESTÃO PARA ATUALIZAR O configs_datasets.json
# ============================================================

print("Sugestões de hiperparâmetros por configuração sensível:\n")

for _, row in df_melhores.iterrows():
    print("-" * 80)
    print(f"Configuração: {row['config_sensivel']}")
    print(f"lambda_fagtb: {row['lambda_fagtb']}")
    print(f"n_estimators: {int(row['n_estimators'])}")
    print(f"learning_rate: {row['learning_rate']}")
    print(f"max_depth: {int(row['max_depth'])}")
    print(f"max_features: {row['max_features']}")
    print(f"min_samples_split: {int(row['min_samples_split'])}")
    print(f"min_impurity: {row['min_impurity']}")
    print(f"balanced_accuracy: {row['balanced_accuracy']:.4f}")
    print(f"accuracy: {row['accuracy']:.4f}")
    print(f"equalized_odds: {row['equalized_odds']:.4f}")
    print(f"score: {row['score']:.4f}")

print("\nArquivos salvos em:")
print(PASTA_SAIDA)


Sugestões de hiperparâmetros por configuração sensível:

--------------------------------------------------------------------------------
Configuração: african_american_0
lambda_fagtb: 0.01
n_estimators: 200
learning_rate: 0.1
max_depth: 3
max_features: nan
min_samples_split: 2
min_impurity: 0
balanced_accuracy: 0.6876
accuracy: 0.8447
equalized_odds: 0.0141
score: 0.6865
--------------------------------------------------------------------------------
Configuração: caucasian_1
lambda_fagtb: 0.01
n_estimators: 200
learning_rate: 0.1
max_depth: 3
max_features: nan
min_samples_split: 2
min_impurity: 0
balanced_accuracy: 0.6889
accuracy: 0.8467
equalized_odds: 0.0098
score: 0.6882

Arquivos salvos em:
resultados\testes_hiperparametros_fagtb
